# 01 — Data exploration (pipeline)

Quick gate before `02_gtfs_processing`: confirm `configs/san_diego.yaml`, raw paths, and latest download manifest. Exports **`pipeline__01_raw_inputs_status__`**, **`pipeline__01_config_snapshot__`**, and **`pipeline__01_latest_manifest_sources__`** under `artifacts/tables/` (dated filenames).

**Pipeline:** run **01 → 07** here after `notebooks/eda/`.

**Previous:** [`eda/`](eda/) · **Next:** [`02_gtfs_processing.ipynb`](02_gtfs_processing.ipynb)

In [3]:
from __future__ import annotations

import json
import sys
import warnings
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import display

# Resolve repo root without importing `src` (so we can add it to sys.path).
_start = Path.cwd().resolve()
REPO_ROOT = next(
    (d for d in [_start, *_start.parents] if (d / "configs" / "san_diego.yaml").exists()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Could not find configs/san_diego.yaml. cd to the repo root or start Jupyter there."
    )
sys.path.insert(0, str(REPO_ROOT))

from src.utils.config import artifact_run_id, load_merged_config  # noqa: E402


def file_stat(rel: Path) -> tuple[bool, int | None]:
    p = REPO_ROOT / rel
    if not p.is_file():
        return False, None
    return True, p.stat().st_size


CONFIG = load_merged_config(REPO_ROOT)
RUN_ID = artifact_run_id()
r5cfg = CONFIG.get("r5") or {}
dep_date = datetime.strptime(str(r5cfg.get("departure_date", "2025-04-15")), "%Y-%m-%d").date()

rows: list[dict] = []
for ag in CONFIG.get("gtfs_agencies") or []:
    ag_root = REPO_ROOT / ag["raw_path"].strip().rstrip("/\\").replace("\\", "/")
    for name, sub in [("feed_info", "extracted/feed_info.txt"), ("stops", "extracted/stops.txt")]:
        abs_p = ag_root / sub
        rel_s = str(abs_p.relative_to(REPO_ROOT)).replace("\\", "/")
        ok, sz = file_stat(Path(rel_s))
        note = ""
        if ok and name == "feed_info":
            try:
                fi = pd.read_csv(abs_p, dtype=str)
                row0 = fi.iloc[0] if len(fi) else None
                if row0 is not None:
                    fs = str(row0.get("feed_start_date", "") or "").strip()
                    fe = str(row0.get("feed_end_date", "") or "").strip()
                    if fs and fe and len(fs) >= 8 and len(fe) >= 8:
                        sdt = datetime.strptime(fs[:8], "%Y%m%d").date()
                        edt = datetime.strptime(fe[:8], "%Y%m%d").date()
                        inside = sdt <= dep_date <= edt
                        note = (
                            f"feed {fs[:8]}–{fe[:8]}; departure {dep_date} "
                            f"{'IN' if inside else 'OUTSIDE'} window"
                        )
                        if not inside:
                            warnings.warn(
                                f"{ag['id']}: departure {dep_date} outside GTFS feed window "
                                f"{sdt}–{edt}.",
                                RuntimeWarning,
                                stacklevel=2,
                            )
            except Exception as ex:  # noqa: BLE001
                note = f"feed_info parse error: {ex}"
        rows.append(
            {
                "component": f"gtfs:{ag['id']}:{name}",
                "path": rel_s,
                "exists": ok,
                "size_bytes": sz if ok else None,
                "note": note,
            }
        )

census_dir = REPO_ROOT / "data" / "raw" / "census"
zips = sorted(census_dir.glob("tl_*_06_tract.zip"))
if zips:
    zp = zips[-1]
    rel_z = str(zp.relative_to(REPO_ROOT)).replace("\\", "/")
    ok, sz = file_stat(Path(rel_z))
    rows.append(
        {
            "component": "census:tiger_zip",
            "path": rel_z,
            "exists": ok,
            "size_bytes": sz if ok else None,
            "note": "",
        }
    )
else:
    rows.append(
        {
            "component": "census:tiger_zip",
            "path": "data/raw/census/tl_*_06_tract.zip",
            "exists": False,
            "size_bytes": None,
            "note": "no matching zip",
        }
    )

shps = sorted(census_dir.glob("tl_*_06_tract/tl_*_06_tract.shp"))
if shps:
    sp = shps[-1]
    rel_s = str(sp.relative_to(REPO_ROOT)).replace("\\", "/")
    ok, sz = file_stat(Path(rel_s))
    rows.append(
        {
            "component": "census:tiger_shp",
            "path": rel_s,
            "exists": ok,
            "size_bytes": sz if ok else None,
            "note": "",
        }
    )
else:
    rows.append(
        {
            "component": "census:tiger_shp",
            "path": "data/raw/census/tl_*_06_tract/tl_*_06_tract.shp",
            "exists": False,
            "size_bytes": None,
            "note": "no shapefile",
        }
    )

acs_files = sorted(census_dir.glob("acs5_*_sd_county.json"))
if acs_files:
    ap = acs_files[-1]
    rel_a = str(ap.relative_to(REPO_ROOT)).replace("\\", "/")
    ok, sz = file_stat(Path(rel_a))
    rows.append(
        {
            "component": "census:acs",
            "path": rel_a,
            "exists": ok,
            "size_bytes": sz if ok else None,
            "note": "",
        }
    )
else:
    rows.append(
        {
            "component": "census:acs",
            "path": "data/raw/census/acs5_*_sd_county.json",
            "exists": False,
            "size_bytes": None,
            "note": "no ACS json",
        }
    )

ok_g, sz_g = file_stat(Path("data/raw/osm/sd_walk_network.graphml"))
rows.append(
    {
        "component": "osm:walk_graph_eda",
        "path": "data/raw/osm/sd_walk_network.graphml",
        "exists": ok_g,
        "size_bytes": sz_g if ok_g else None,
        "note": "osmnx/EDA walk network (GraphML)",
    }
)

pbf_rel = str(r5cfg.get("osm_pbf", "data/raw/osm/san_diego_study.osm.pbf")).replace("\\", "/")
ok_pbf, sz_pbf = file_stat(Path(pbf_rel))
rows.append(
    {
        "component": "osm:r5_pbf",
        "path": pbf_rel,
        "exists": ok_pbf,
        "size_bytes": sz_pbf if ok_pbf else None,
        "note": "required for r5py (notebook 03)",
    }
)

lodes_dir = REPO_ROOT / "data/raw/external/lodes"
if lodes_dir.is_dir():
    lodes_files = sorted(lodes_dir.glob("ca_wac_*.csv.gz"))
    if lodes_files:
        for f in lodes_files:
            rel_s = str(f.relative_to(REPO_ROOT)).replace("\\", "/")
            rows.append(
                {
                    "component": "lodes:wac",
                    "path": rel_s,
                    "exists": True,
                    "size_bytes": f.stat().st_size,
                    "note": "aligned glob with notebook 03",
                }
            )
    else:
        rows.append(
            {
                "component": "lodes:wac",
                "path": "data/raw/external/lodes/ca_wac_*.csv.gz",
                "exists": False,
                "size_bytes": None,
                "note": "no ca_wac_*.csv.gz",
            }
        )
else:
    rows.append(
        {
            "component": "lodes:wac",
            "path": "data/raw/external/lodes/",
            "exists": False,
            "size_bytes": None,
            "note": "",
        }
    )

df = pd.DataFrame(rows)
art = REPO_ROOT / "artifacts" / "tables"
art.mkdir(parents=True, exist_ok=True)
p_status = art / f"pipeline__01_raw_inputs_status__{RUN_ID}.csv"
df.to_csv(p_status, index=False)

acc = CONFIG.get("accessibility", {})
mod = CONFIG.get("model", {})
exp = CONFIG.get("export", {})
snap = pd.DataFrame(
    [
        {
            "run_id": RUN_ID,
            "city": CONFIG.get("city"),
            "city_label": CONFIG.get("city_label"),
            "county_fips": CONFIG.get("county_fips"),
            "bbox": str(CONFIG.get("bbox")),
            "n_gtfs_agencies": len(CONFIG.get("gtfs_agencies") or []),
            "config_merged_from": "configs/defaults.yaml + configs/san_diego.yaml",
            "travel_time_threshold_min": acc.get("travel_time_threshold_min"),
            "walk_speed_kph": acc.get("walk_speed_kph"),
            "departure_window_start": acc.get("departure_window_start"),
            "departure_window_end": acc.get("departure_window_end"),
            "opportunities": str(acc.get("opportunities")),
            "model_sampler": mod.get("sampler"),
            "model_draws": mod.get("draws"),
            "model_tune": mod.get("tune"),
            "model_chains": mod.get("chains"),
            "model_target_accept": mod.get("target_accept"),
            "r5_departure_date": r5cfg.get("departure_date"),
            "r5_departure_hhmm": r5cfg.get("departure_hhmm"),
            "r5_osm_pbf": r5cfg.get("osm_pbf"),
            "export_geojson_precision": exp.get("geojson_precision"),
        }
    ]
)
p_snap = art / f"pipeline__01_config_snapshot__{RUN_ID}.csv"
snap.to_csv(p_snap, index=False)

missing_required = df[(~df["exists"]) & df["component"].isin(["osm:r5_pbf", "lodes:wac"])]
if not missing_required.empty:
    warnings.warn(
        "BLOCKING: required inputs missing for pipeline 03:\n"
        + missing_required[["component", "path", "note"]].to_string(index=False),
        RuntimeWarning,
        stacklevel=2,
    )

prov_dir = REPO_ROOT / "artifacts" / "logs" / "provenance"
manifests = sorted(prov_dir.glob("data_manifest_*.json"), reverse=True)
latest_manifest = manifests[0] if manifests else None
if latest_manifest:
    meta = json.loads(latest_manifest.read_text(encoding="utf-8"))
    dfm = pd.DataFrame(meta.get("sources", []))
    p_man = art / f"pipeline__01_latest_manifest_sources__{RUN_ID}.csv"
    dfm.to_csv(p_man, index=False)

display(snap)
display(df)
if latest_manifest:
    display(dfm.head(20))


c:\Users\sardo\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3577: RuntimeWarning: mts: departure 2025-04-15 outside GTFS feed window 2026-01-25–2026-06-06.
  exec(code_obj, self.user_global_ns, self.user_ns)
c:\Users\sardo\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3577: RuntimeWarning: BLOCKING: required inputs missing for pipeline 03:
 component                                 path                            note
osm:r5_pbf data/raw/osm/san_diego_study.osm.pbf required for r5py (notebook 03)
  exec(code_obj, self.user_global_ns, self.user_ns)


,run_id,city,city_label,county_fips,bbox,n_gtfs_agencies,config_merged_from,travel_time_threshold_min,walk_speed_kph,departure_window_start,...,opportunities,model_sampler,model_draws,model_tune,model_chains,model_target_accept,r5_departure_date,r5_departure_hhmm,r5_osm_pbf,export_geojson_precision
0,2026-03-30,san_diego,"San Diego, CA",073,"[-117.4, 32.53, -116.8, 33.35]",2,configs/defaults.yaml + configs/san_diego.yaml,45,4.8,07:00,...,"['jobs', 'hospitals', 'groceries', 'schools']",mcmc,2000,1000,4,0.9,2025-04-15,07:30,data/raw/osm/san_diego_study.osm.pbf,5


,component,path,exists,size_bytes,note
0,gtfs:mts:feed_info,data/raw/gtfs/mts/extracted/feed_info.txt,True,222.0,feed 20260125–20260606; departure 2025-04-15 O...
1,gtfs:mts:stops,data/raw/gtfs/mts/extracted/stops.txt,True,426107.0,
2,gtfs:nctd:feed_info,data/raw/gtfs/nctd/extracted/feed_info.txt,True,111.0,
3,gtfs:nctd:stops,data/raw/gtfs/nctd/extracted/stops.txt,True,179449.0,
4,census:tiger_zip,data/raw/census/tl_2023_06_tract.zip,True,32523329.0,
5,census:tiger_shp,data/raw/census/tl_2023_06_tract/tl_2023_06_tr...,True,51594156.0,
6,census:acs,data/raw/census/acs5_2023_sd_county.json,True,139135.0,
7,osm:walk_graph_eda,data/raw/osm/sd_walk_network.graphml,True,383452356.0,osmnx/EDA walk network (GraphML)
8,osm:r5_pbf,data/raw/osm/san_diego_study.osm.pbf,False,NaN,required for r5py (notebook 03)
9,lodes:wac,data/raw/external/lodes/ca_wac_2021.csv.gz,True,6000959.0,aligned glob with notebook 03


,label,source_type,tool,bbox,network_type,n_nodes,n_edges,dest,status,md5,size_bytes,downloaded_at,note
0,OSM Walk Network,osm_pedestrian_network,osmnx,"[-117.4, 32.53, -116.8, 33.35]",walk,337824,906988,data\raw\osm\sd_walk_network.graphml,downloaded,733d726d74882f3068ceb6ecb2200a99,383452356,2026-03-30T18:44:26.550777+00:00,OSM has no versioning. Timestamp is the only p...
